# FX News Impact Triage agent using Jev/OpenJev
This is a research and alerting system that takes a news headline, a source and a currency and assesses the impact of the price. 

!Notes: This is not a trade execution agent, this is purely for experimental purposes.

In [ ]:
# Install dependencies
# %pip install python-dotenv typesafe_sdk
from dotenv import load_dotenv
import os
load_dotenv()
from typesafe_sdk import TypeSafeClient
MODEL_NAME="jev-latest"

In [14]:
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score
from pydantic import BaseModel, ConfigDict, Field, field_validator
class FxImpactFacts(BaseModel):
    model: str
    policy_signal: float
    currency_impact: str
    currency_impact_confidence: float
    materiality: float

def headline_triage(currency, context) -> dict:
    with TypeSafeClient() as client:
        response = client.system_one(
            state= context,
            questions={
                "policy_signal": Noul(
                    instructions=(
                        "The statement signals concern about persistent "
                        "inflation or a need to maintain restrictive policy."
                    )
                ),
                "currency_impact": Choice(
                    instructions=(
                        f"What is the likely directional implication for {currency}, "
                        "considering monetary policy expectations?"
                    ),
                    criteria={
                        f"{currency}_BULLISH": "Likely supportive of {currency}",
                        f"{currency}_BEARISH": "Likely negative for {currency}",
                        "NEUTRAL": "No meaningful directional impact",
                        "UNCLEAR": "Insufficient or conflicting evidence",
                    },
                ),
            "materiality": Score(
                instructions=(
                    "How material is this headline for near-term G10 FX markets?"
                ),
                criteria=[
                    "Negligible",
                    "Low",
                    "Moderate",
                    "High",
                    "Very high",
                ],
            ),
        },
        model=MODEL_NAME,
    )

    # print("Model:", response.model)
    # print("Policy signal:", response.nouls["policy_signal"].noul)
    # print("Direction:", response.choices["currency_impact"].choice)
    # print("Direction confidence:", response.choices["currency_impact"].confidence)
    # print("Materiality score:", response.scores["materiality"].score)
    return FxImpactFacts(
        model=response.model,
        policy_signal=response.nouls["policy_signal"].noul,
        currency_impact=response.choices["currency_impact"].choice,
        currency_impact_confidence=response.choices["currency_impact"].confidence,
        materiality=response.scores["materiality"].score,
    )

In [19]:
arr_headlines = [
    { "currency": "USD", "context": "The Federal Reserve says inflation remains persistent."},
    { "currency": "INR", "context": "The Federal Reserve says inflation remains persistent."},
    { "currency": "GBP", "context": "The Federal Reserve says inflation remains persistent."},
    { "currency": "INR", "context": "The Reserve Bank of India says inflation remains persistent."},
    { "currency": "GBP", "context": "The Federal Reserve says inflation remains persistent."},
    { "currency": "USD", "context": "The Federal Reserve says inflation remains persistent."},
    { "currency": "USD", "context": "Russia resuming talks with Ukraine."},
    { "currency": "GBP", "context": "Russia resuming talks with Ukraine."},
    { "currency": "INR", "context": "Russia resuming talks with Ukraine."},
    { "currency": "USD", "context": "Crude oil rises above $120 per barrel."},
    { "currency": "GBP", "context": "Crude oil rises above $120 per barrel."},
    { "currency": "INR", "context": "Crude oil rises above $120 per barrel."},
]
rows = []
for headline in arr_headlines:
    headline_facts = headline_triage(headline["currency"], headline["context"])
    rows.append({
        "currency": headline["currency"],
        "headline": headline["context"],
        "model": headline_facts.model,
        "policy_signal": headline_facts.policy_signal,
        "currency_impact": headline_facts.currency_impact,
        "currency_impact_confidence": headline_facts.currency_impact_confidence,
        "materiality": headline_facts.materiality,
    })

In [20]:
import pandas as pd
output =pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
output

,currency,headline,model,policy_signal,currency_impact,currency_impact_confidence,materiality
0,USD,The Federal Reserve says inflation remains persistent.,jev-1.13.0,0.86,USD_BULLISH,1.00,2.66
1,INR,The Federal Reserve says inflation remains persistent.,jev-1.13.0,0.87,INR_BEARISH,0.60,2.64
2,GBP,The Federal Reserve says inflation remains persistent.,jev-1.13.0,0.87,UNCLEAR,0.16,2.66
3,INR,The Reserve Bank of India says inflation remains persistent.,jev-1.13.0,0.85,INR_BEARISH,0.49,1.22
4,GBP,The Federal Reserve says inflation remains persistent.,jev-1.13.0,0.87,UNCLEAR,0.17,2.65
5,USD,The Federal Reserve says inflation remains persistent.,jev-1.13.0,0.86,USD_BULLISH,1.00,2.65
6,USD,Russia resuming talks with Ukraine.,jev-1.13.0,0.04,UNCLEAR,0.33,2.37
7,GBP,Russia resuming talks with Ukraine.,jev-1.13.0,0.05,UNCLEAR,0.44,2.44
8,INR,Russia resuming talks with Ukraine.,jev-1.13.0,0.05,UNCLEAR,0.24,2.43
9,USD,Crude oil rises above $120 per barrel.,jev-1.13.0,0.74,USD_BULLISH,0.61,2.77
